CI Bus Routing Experimentation - Optimizer Experimentation

Part 1: Set Up Codespace

In [1]:
import gymnasium as gym
import numpy as np
import numpy as np
import torch
import torch.nn as nn
from random import sample
from gymnasium import spaces
from collections import deque
from matplotlib import pyplot as plt

In [2]:
# Check for GPU availability (CUDA first, then MPS, then CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


Part 2: Custom Gymnasium Environment

In [3]:
class BusEnv(gym.Env):
    metadata = {"render_modes": ["human"], "render_fps": 4}

    def __init__(self, config=None, render_mode=None):
        super().__init__()

        config = config or {}

        # Route / simulation settings
        self.num_stops = config.get("num_stops", 14)
        self.max_buses = config.get("max_buses", 6)
        self.bus_capacity = config.get("bus_capacity", 100)
        self.episode_length = config.get("episode_length", 960)  # 8:00am to 12:00am, 1 step = 1 minute
        self.max_queue = config.get("max_queue", 500)

        self.base_arrival_rates = np.array(
            config.get(
                "base_arrival_rates",
                [30, 2, 2, 2, 2, 2, 2, 40, 4, 2, 2, 2, 2, 2],
            ),
            dtype=np.float32,
        )

        # Reward weights
        self.pickup_reward = config.get("pickup_reward", 0.1)
        self.left_behind_penalty = config.get("left_behind_penalty", 0.001)
        self.waiting_penalty = config.get("waiting_penalty", 0.00001)
        self.active_bus_penalty = config.get("active_bus_penalty", 0.1)

        self.render_mode = render_mode
        assert render_mode is None or render_mode in self.metadata["render_modes"]

        # Bus features per bus:
        # stop, occupancy, active, held, segment
        self.bus_features = 5

        # Observation:
        # [timestep, active_bus_count]
        # + queue lengths for each stop
        # + time since last service for each stop
        # + bus states for each possible bus
        obs_size = (
            2 + self.num_stops + self.num_stops + self.max_buses * self.bus_features
        )
        self.observation_space = spaces.Box(
            low=0.0,
            high=1000.0,
            shape=(obs_size,),
            dtype=np.float32,
        )

        # 0 = normal operation
        # 1 = add a bus
        # 2 = remove a bus
        # 3 = hold a bus
        self.action_space = spaces.Discrete(4)

        self.window = None
        self.clock = None

        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.timestep = 0

        # Queue at each stop
        self.queues = np.zeros(self.num_stops, dtype=np.int32)

        # How many steps since each stop was last served
        self.last_service = np.zeros(self.num_stops, dtype=np.int32)

        # Bus states
        self.buses = []
        for _ in range(self.max_buses):
            self.buses.append(
                {
                    "stop": 0,
                    "occupancy": 0,
                    "active": 0,
                    "held": 0,
                }
            )

        # Start with one active bus at stop 0
        self.buses[0]["active"] = 1
        self.buses[0]["stop"] = 0
        self.buses[0]["occupancy"] = 0
        self.buses[0]["held"] = 0

        obs = self._get_obs()
        info = self._get_info()

        return obs, info

    def step(self, action):
        self.timestep += 1

        self._generate_passengers()
        self._apply_action(action)
        picked_up, left_behind = self._move_buses_and_board()

        # Increase time since last service for all stops,
        # then reset happens inside boarding when a stop is served
        self.last_service += 1

        total_waiting = int(np.sum(self.queues))
        active_buses = self._active_bus_count()

        reward = (
            self.pickup_reward * picked_up
            - self.left_behind_penalty * left_behind
            - self.waiting_penalty * total_waiting
            - self.active_bus_penalty * active_buses
        )

        terminated = self.timestep >= self.episode_length
        truncated = False

        obs = self._get_obs()
        info = self._get_info()
        info.update(
            {
                "picked_up": picked_up,
                "left_behind": left_behind,
                "reward": reward,
            }
        )

        if self.render_mode == "human":
            self.render()

        return obs, reward, terminated, truncated, info

    def render(self):
        print(f"\nTime step: {self.timestep}")
        print(f"Queues: {self.queues.tolist()}")
        for i, bus in enumerate(self.buses):
            if bus["active"]:
                print(
                    f"Bus {i}: stop={bus['stop']}, "
                    f"segment={'WB' if self.is_westbound_stop(bus['stop']) else 'EB'}, "
                    f"occupancy={bus['occupancy']}, held={bus['held']}"
                )

    def close(self):
        pass

    def _get_obs(self):
        obs = [self.timestep, self._active_bus_count()]

        # stop queues
        obs.extend(self.queues.tolist())

        # time since last service
        obs.extend(self.last_service.tolist())

        # per-bus features
        for bus in self.buses:
            segment = 0 if self.is_westbound_stop(bus["stop"]) else 1
            obs.extend(
                [
                    bus["stop"],
                    bus["occupancy"],
                    bus["active"],
                    bus["held"],
                    segment,
                ]
            )

        return np.array(obs, dtype=np.float32)

    def _get_info(self):
        return {
            "timestep": self.timestep,
            "active_buses": self._active_bus_count(),
            "total_waiting": int(np.sum(self.queues)),
        }

    def _generate_passengers(self):
        arrivals = self.np_random.poisson(self.base_arrival_rates).astype(np.int32)

        # 9:00am class spike
        # If 8:00 = step 0, then 9:00 = step 60
        if 55 <= self.timestep <= 70:
            arrivals[0] += self.np_random.poisson(150)  # East campus terminal spike

        # 11:30am spike
        if 205 <= self.timestep <= 220:
            arrivals[7] += self.np_random.poisson(150)  # West campus terminal spike

        self.queues = np.minimum(self.queues + arrivals, self.max_queue)

    def _apply_action(self, action):
        if action == 1:
            self._add_bus()
        elif action == 2:
            self._remove_bus()
        elif action == 3:
            self._hold_bus()
        # action == 0 means normal operation

    def _add_bus(self):
        inactive_indices = [i for i, b in enumerate(self.buses) if b["active"] == 0]
        if not inactive_indices:
            return

        # Place new bus at busiest stop
        busiest_stop = int(np.argmax(self.queues))
        idx = inactive_indices[0]

        self.buses[idx]["active"] = 1
        self.buses[idx]["stop"] = busiest_stop
        self.buses[idx]["occupancy"] = 0
        self.buses[idx]["held"] = 0

    def _remove_bus(self):
        active_indices = [i for i, b in enumerate(self.buses) if b["active"] == 1]

        # Keep at least one bus active
        if len(active_indices) <= 1:
            return

        # Remove emptiest active bus
        remove_idx = min(active_indices, key=lambda i: self.buses[i]["occupancy"])

        self.buses[remove_idx]["active"] = 0
        self.buses[remove_idx]["occupancy"] = 0
        self.buses[remove_idx]["held"] = 0

    def _hold_bus(self):
        active_indices = [i for i, b in enumerate(self.buses) if b["active"] == 1]
        if not active_indices:
            return

        # Hold the bus closest to the busiest stop
        busiest_stop = int(np.argmax(self.queues))

        def loop_distance(a, b):
            forward = (b - a) % self.num_stops
            backward = (a - b) % self.num_stops
            return min(forward, backward)

        hold_idx = min(
            active_indices,
            key=lambda i: loop_distance(self.buses[i]["stop"], busiest_stop),
        )
        self.buses[hold_idx]["held"] = 1

    def _move_buses_and_board(self):
        total_picked_up = 0
        total_left_behind = 0

        for bus in self.buses:
            if bus["active"] == 0:
                continue

            # Move unless held
            if bus["held"] == 1:
                bus["held"] = 0
            else:
                bus["stop"] = self.next_stop(bus["stop"])

            stop = bus["stop"]

            # Board passengers
            waiting = int(self.queues[stop])
            available_space = self.bus_capacity - bus["occupancy"]

            boarded = min(waiting, available_space)
            left_behind = max(0, waiting - available_space)

            bus["occupancy"] += boarded
            self.queues[stop] -= boarded
            self.last_service[stop] = 0

            total_picked_up += boarded
            total_left_behind += left_behind

            # Simple drop-off rule:
            if stop == 0 or stop == 7:
                dropped = bus["occupancy"]  # All off at terminals
            else:
                dropped = int(
                    0.01 * bus["occupancy"]
                )  # some fraction of riders leave at each stop
            bus["occupancy"] -= dropped

        return total_picked_up, total_left_behind

    def next_stop(self, stop):
        return (stop + 1) % self.num_stops

    def is_westbound_stop(self, stop):
        return 0 <= stop <= 7

    def is_eastbound_stop(self, stop):
        return 8 <= stop <= 13

    def _active_bus_count(self):
        return sum(bus["active"] for bus in self.buses)

Part 3: Neural Net and Agent

In [4]:
class NeuralNet(torch.nn.Module):
    """
    Implements a neural network representation of
    the Q-function for use in DQN.
    """

    def __init__(self):
        super(NeuralNet, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(60, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 4),
        )

    def forward(self, x):
        """
        Input should represent state/observation space encoding
        Output should be a q-function estimate for each possible
        discrete action.
        """
        logits = self.net(x)
        return logits


class BusAgent:
    """
    Implements a deep Q-learning agent for the C1 Bus environment.
    """

    def __init__(
        self,
        env,
        discount=0.95,
        learning_rate=0.001,
        buffer_size=100000,
        batch_size=64,
        target_update_freq=1000,
        epsilon_start=1.0,
        epsilon_min=0.01,
        epsilon_decay=0.99999,
        optimizer_class=torch.optim.Adam,
        optimizer_kwargs=None
    ):
        """
        Initialize the DQN agent.

        Args:
            discount: Discount factor (gamma)
            learning_rate: Learning rate for optimizer
            buffer_size: Maximum size of replay buffer
            batch_size: Number of transitions to sample per update
            target_update_freq: Steps between target network updates
            epsilon_start: Initial exploration rate
            epsilon_min: Minimum exploration rate
            epsilon_decay: Multiplicative decay for epsilon
        """
        self.env = env
        self.discount = discount
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.epsilon = epsilon_start
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay

        # Initialize replay buffer
        self.replay_buffer = deque(maxlen=buffer_size)

        # Initialize networks, loss, and optimizer
        self.main_q = NeuralNet().to(device)
        self.target_q = NeuralNet().to(device)

        self.target_q.load_state_dict(self.main_q.state_dict())
        self.target_q.eval()
        self.step_count = 0

        if optimizer_kwargs is None:
            optimizer_kwargs = {}

        self.optimizer = optimizer_class(
                self.main_q.parameters(),
                lr=learning_rate,
                **optimizer_kwargs
            )
        self.loss = nn.MSELoss()

    def action_select(self, state):
        """
        Epsilon-greedy action selection using neural network.

        Args:
            state: NumPy array of shape (8,)

        Returns:
            action: Integer action (0-3)
        """
        if np.random.rand() < self.epsilon:
            return self.env.action_space.sample()

        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)

        with torch.no_grad():
            q_values = self.main_q(state_tensor)

        return torch.argmax(q_values, dim=1).item()

    def update(self, state, action, reward, next_state, terminated):
        """
        Store experience and perform learning update if buffer is ready.

        Args:
            state: Current state
            action: Action taken
            reward: Reward received
            next_state: Next state
            terminated: Whether episode terminated
        """
        self.replay_buffer.append((state, action, reward, next_state, terminated))

        if len(self.replay_buffer) < self.batch_size:
            return

        batch = sample(self.replay_buffer, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        states = torch.tensor(np.array(states), dtype=torch.float).to(device)
        actions = (
            torch.tensor(np.array(actions), dtype=torch.long).unsqueeze(1).to(device)
        )
        rewards = (
            torch.tensor(np.array(rewards), dtype=torch.float).unsqueeze(1).to(device)
        )
        next_states = torch.tensor(np.array(next_states), dtype=torch.float).to(device)
        terminations = (
            torch.tensor(np.array(dones), dtype=torch.int).unsqueeze(1).to(device)
        )

        q_values = self.main_q(states).gather(1, actions)

        with torch.no_grad():
            max_next_q = self.target_q(next_states).max(1, keepdim=True)[0]
            targets = rewards + (1 - terminations) * self.discount * max_next_q

        loss = self.loss(q_values, targets)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.step_count += 1

        if self.step_count % self.target_update_freq == 0:
            self.target_q.load_state_dict(self.main_q.state_dict())
            
    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)



Part 4: Training Loop

In [ ]:
def train(agent, env, n_episodes=1000, print_every=10, timestep_penalty=0.2):
    """
    Train a Q-learning agent.

    Args:
        agent: BusAgent instance to train
        env: Gymnasium environment (should be wrapped with RecordEpisodeStatistics)
        n_episodes: Number of episodes to train for
        print_every: Print progress every N episodes (set to None to disable printing)

    Returns:
        agent: The trained agent
        env: The environment with recorded statistics
    """
    for episode in range(n_episodes):
        state, info = env.reset()
        episode_over = False

        while not episode_over:
            action = agent.action_select(state)

            next_state, reward, terminated, truncated, _ = env.step(action)

            agent.update(state, action, reward, next_state, terminated)

            state = next_state

            episode_over = terminated or truncated
        
        agent.decay_epsilon()
        
        if print_every and (episode + 1) % print_every == 0:
            print(
                f"Episode {episode}, Average Reward: {np.mean(list(env.return_queue)[-100:])}"
            )

    return agent, env

n_episodes = 600

# Register the environment so we can create it with gym.make()
gym.register(
    id="gymnasium_env/BusRouting-v0",
    entry_point=BusEnv,
    max_episode_steps=300,  # Prevent infinite episodes
)

envs = []

# Create the environment like any built-in environment
env_adam = gym.make("gymnasium_env/BusRouting-v0")
env_adam = gym.wrappers.RecordEpisodeStatistics(env_adam, buffer_length=n_episodes)

env_adamw = gym.make("gymnasium_env/BusRouting-v0")
env_adamw = gym.wrappers.RecordEpisodeStatistics(env_adamw, buffer_length=n_episodes)
    
env_RMSprop = gym.make("gymnasium_env/BusRouting-v0")
env_RMSprop = gym.wrappers.RecordEpisodeStatistics(env_RMSprop, buffer_length=n_episodes)
    
# Can adjust training hyperparameters of agent as needed
agent_adam = BusAgent(env_adam)
agent_adamw = BusAgent(env_adamw, optimizer_class = torch.optim.AdamW)
agent_RMSprop = BusAgent(env_RMSprop, optimizer_class = torch.optim.RMSprop)

# Train the agent
agent_adam, env_adam = train(agent_adam, env_adam, n_episodes=n_episodes, print_every=20)
    
print(f"\nTraining of Agent_Adam complete!")
print(
    f"Final average return (last 100 episodes): {np.mean(list(env_adam.return_queue)[-100:]):.2f}"
)
    
agent_adamw, env_adamw = train(agent_adamw, env_adamw, n_episodes=n_episodes, print_every=20)
    
print(f"\nTraining of Agent_AdamW complete!")
print(
    f"Final average return (last 100 episodes): {np.mean(list(env_adamw.return_queue)[-100:]):.2f}"
)
    
agent_RMSprop, env_RMSprop = train(agent_RMSprop, env_RMSprop, n_episodes=n_episodes, print_every=20)
    
print(f"\nTraining of Agent_RMSprop complete!")
print(
    f"Final average return (last 100 episodes): {np.mean(list(env_RMSprop.return_queue)[-100:]):.2f}"
)

Episode 19, Average Reward: 1278.4926020000003
Episode 39, Average Reward: 1195.3854990000002
Episode 59, Average Reward: 1204.6570466666672
Episode 79, Average Reward: 1212.987492375
Episode 99, Average Reward: 1214.3394998
Episode 119, Average Reward: 1196.5686981000001
Episode 139, Average Reward: 1209.3007561
Episode 159, Average Reward: 1216.9282433
Episode 179, Average Reward: 1206.3719776
Episode 199, Average Reward: 1205.5811319
Episode 219, Average Reward: 1221.8728975
Episode 239, Average Reward: 1233.3393591
Episode 259, Average Reward: 1221.7691652
Episode 279, Average Reward: 1227.9248596
Episode 299, Average Reward: 1226.3766721000002
Episode 319, Average Reward: 1228.6052966000002
Episode 339, Average Reward: 1223.9257364000002
Episode 359, Average Reward: 1228.8715111
Episode 379, Average Reward: 1230.1979181999998
Episode 399, Average Reward: 1237.9270447000001
Episode 419, Average Reward: 1231.5383296
Episode 439, Average Reward: 1230.8472055000002
Episode 459, Averag

In [ ]:
def visualize_learning_curves(envs, smoothing=10, labels=None):
    fig, ax = plt.subplots(1, 1, figsize=(20, 8))

    if labels is None:
        labels = [f"Run {i}" for i in range(len(envs))]

    for env, label in zip(envs, labels):
        returns = np.array(env.return_queue)
        returns_smoothed = np.convolve(
            returns, np.ones(smoothing)/smoothing, mode='valid'
        )

        ax.plot(returns_smoothed, label=label)

    ax.set_title("Episode Returns", fontsize=20)
    ax.set_xlabel("Episode", fontsize=20)
    ax.set_ylabel("Return", fontsize=20)
    ax.legend()

    plt.tight_layout()
    plt.show()
    
visualize_learning_curves(envs=[env_adamw, env_adam, env_RMSprop], smoothing=10, labels=["AdamW", "Adam", "RMSprop"])